# CF0360 all-species DA, May 2019: soil moisture states (SFMC, RZMC)

Companion to `cf0360_allsp_may2019_ofa_maps.ipynb` (observation space). This notebook looks at the **model
states**: surface (`SFMC`, 0–5 cm) and root-zone (`RZMC`, 0–100 cm) soil moisture, in m³/m³, for the
all-species DA run (`allsp`) and the 6-member open loop (`OL6`).

| Name here | Experiment |
|---|---|
| `allsp` | `M21C_test_CF0360_allsp_superob025ml_OL6sc_xc03125` (24 members, all species assimilated) |
| `OL6` | `M21C_test_CF0360_OL6_superob025ml_clim` (6 members, no assimilation) |

**What output is available, and why that shapes this notebook:**

- **OL6 wrote only the daily 0.5° gridded collection** `tavg24_2d_lnd_Nx`. It is made from tile values in two steps:
  a `frac_cell`-weighted mean of the land tiles per cube cell, then bilinear interpolation to 0.5°
  (see `cf0360_grid_and_tiles.ipynb`, section 7). So **allsp vs OL6 can only be compared at 0.5°**, using the same
  collection from both runs. It is a like-for-like comparison, but single-tile features (e.g. the saturated
  Pantanal wetland tiles) are diluted.
- **allsp also wrote tile-space fields** every 3 h: `inst3_1d_lndfcstana_Nt` holds the ensemble-mean forecast
  (`*_FCST`) and analysis (`*_ANA`) at each assimilation time. Their difference is the **analysis increment**,
  i.e. how much water DA added or removed. Section 5 sums those increments over the month in tile space and
  maps them on the native cube cells.

Sections:

1. Configuration
2. Read the daily 0.5° files of both runs
3. Daily time series: global and latitude-band means, allsp − OL6
4. Maps at 0.5°: OL6 monthly mean, allsp − OL6 monthly mean, allsp − OL6 on 31 May
5. allsp only, native resolution: summed analysis increments per cube cell

Run on Discover with the GEOSpyD Python 3.13 environment. Section 5 reads 248 tile files (a couple of minutes)
and caches the result.

## 1. Imports and configuration

In [ ]:
import glob
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import netCDF4 as nc
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature

%matplotlib inline


def find_repo_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / '.git').exists():
            return p
    return start

REPO_ROOT = find_repo_root(Path.cwd())

ROOT = Path('/gpfsm/dnb06/projects/p284/M21C_testing')
DOMAIN = 'CF0360x6C_GLOBAL'
RUNS = {
    'allsp': 'M21C_test_CF0360_allsp_superob025ml_OL6sc_xc03125',
    'OL6':   'M21C_test_CF0360_OL6_superob025ml_clim',
}
DA_RUN, OL_RUN = 'allsp', 'OL6'
START, END = '20190501', '20190531'          # inclusive, by date
VARS = ['SFMC', 'RZMC']
VAR_LABEL = {'SFMC': 'surface soil moisture (0–5 cm)', 'RZMC': 'root-zone soil moisture (0–100 cm)'}

# Latitude bands for the time series (same bands as the earlier O-F comparisons), south of 60S excluded
LAT_BANDS = [(-60, -30), (-30, 0), (0, 30), (30, 45), (45, 55), (55, 90)]
MAP_LAT_MIN = -60.0

CACHE_DIR = ROOT / 'analysis_M21C_land' / 'allsp_vs_OL6_may2019'
FIG_DIR = REPO_ROOT / 'projects' / 'M21C_land' / 'figures' / 'allsp_vs_OL6_may2019_states'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
FORCE_RECOMPUTE = False

# CF0360 geometry (for section 5): tile table and the cube-cell raster from BCS v14
BCS_GEOM = Path('/discover/nobackup/projects/gmao/bcs_shared/fvInput/ExtData/esm/tiles/v14/geometry/CF0360x6C_CF0360x6C')
N_FACE = 360

## 2. Read the daily 0.5° files (`tavg24_2d_lnd_Nx`) of both runs

One file per day, time-stamped at 12z, holding the mean from 00z to 24z of that day. Fill values (1e15:
ocean, or no land tile nearby) become NaN.

Grid: 720 × 361 cell **centres** at every 0.5°, from −180 and from −90 to 90 (pole-centred). For `pcolormesh`
we build the cell **edges** halfway between centres.

**Area weights** for global and band means: `cos(lat)` per 0.5° cell with a valid value. The file has no
land-fraction field, so a coastal cell counts as much as an inland cell of the same latitude. That is fine for
comparing the two runs, which share the same cells, but it is not an exact land-area mean.

In [ ]:
def read_daily(exp):
    """dict var -> array (n_days, 361, 720) plus the list of dates, for [START, END]."""
    files = sorted(glob.glob(str(ROOT / exp / 'output' / DOMAIN / 'cat' / 'ens_avg' / 'Y*' / 'M*'
                                 / f'{exp}.tavg24_2d_lnd_Nx.*.nc4')))
    files = [f for f in files if START <= os.path.basename(f).split('.')[-2][:8] <= END]
    out = {v: [] for v in VARS}
    dates = []
    for f in files:
        dates.append(pd.Timestamp(os.path.basename(f).split('.')[-2][:8]))
        with nc.Dataset(f) as d:
            for v in VARS:
                out[v].append(np.ma.filled(d[v][0].astype(np.float64), np.nan))
            lat, lon = np.asarray(d['lat'][:]), np.asarray(d['lon'][:])
    return {v: np.stack(a) for v, a in out.items()}, dates, lat, lon


DAILY = {}
for name, exp in RUNS.items():
    DAILY[name], DATES, LAT, LON = read_daily(exp)
    print(f'{name}: {len(DATES)} days, {DATES[0].date()} .. {DATES[-1].date()}')
assert all(DAILY[DA_RUN][v].shape == DAILY[OL_RUN][v].shape for v in VARS)

LAT_E = np.concatenate([[-90.0], 0.5 * (LAT[:-1] + LAT[1:]), [90.0]])
LON_E = np.concatenate([[LON[0] - 0.25], 0.5 * (LON[:-1] + LON[1:]), [LON[-1] + 0.25]])
W2D = np.cos(np.radians(LAT))[:, None] * np.ones(LON.size)[None, :]     # cos(lat) area weight per cell

# Common land mask: cells valid in BOTH runs on EVERY day (so means compare the same cells)
VALID = np.ones(W2D.shape, bool)
for name in RUNS:
    for v in VARS:
        VALID &= np.all(np.isfinite(DAILY[name][v]), axis=0)
VALID &= (LAT >= MAP_LAT_MIN)[:, None]
print(f'common valid 0.5° land cells (north of 60S): {VALID.sum():,}')

## 3. Daily time series

For each day, the area-weighted mean over the common land cells, globally and per latitude band.

- **Top row:** global mean SFMC and RZMC, OL6 (dashed) and allsp (solid).
- **Bottom row:** the difference allsp − OL6 per latitude band. A difference that keeps growing means DA
  steadily adds (or removes) water. A flat difference that appears in the first days is an initial adjustment.

In [ ]:
def area_mean(field3d, mask):
    """Area-weighted mean over mask for each day: field3d (n_days, lat, lon) -> (n_days,)."""
    w = np.where(mask, W2D, 0.0)
    return np.nansum(field3d * w, axis=(1, 2)) / w.sum()


rows = []
for v in VARS:
    for name in RUNS:
        rows.append(pd.DataFrame({'date': DATES, 'var': v, 'run': name, 'band': 'global',
                                  'mean': area_mean(DAILY[name][v], VALID)}))
        for lo, hi in LAT_BANDS:
            band_mask = VALID & ((LAT >= lo) & (LAT < hi))[:, None]
            rows.append(pd.DataFrame({'date': DATES, 'var': v, 'run': name, 'band': f'{lo}..{hi}',
                                      'mean': area_mean(DAILY[name][v], band_mask)}))
TS = pd.concat(rows, ignore_index=True)
TS_W = TS.pivot_table(index=['var', 'band', 'date'], columns='run', values='mean').reset_index()
TS_W['diff'] = TS_W[DA_RUN] - TS_W[OL_RUN]
TS_W.to_csv(CACHE_DIR / 'states_daily_means.csv', index=False)

fig, axs = plt.subplots(2, 2, figsize=(15, 9), sharex=True, constrained_layout=True)
band_colors = plt.get_cmap('viridis')(np.linspace(0, 0.95, len(LAT_BANDS)))
for c, v in enumerate(VARS):
    g = TS_W[(TS_W['var'] == v) & (TS_W.band == 'global')]
    axs[0, c].plot(g.date, g[OL_RUN], 'k--', label=f'{OL_RUN} (mean {g[OL_RUN].mean():.4f})')
    axs[0, c].plot(g.date, g[DA_RUN], 'k-', label=f'{DA_RUN} (mean {g[DA_RUN].mean():.4f})')
    axs[0, c].set_title(f'{v}: global land mean (N of 60S)')
    axs[0, c].set_ylabel(f'{v} [m³/m³]')
    axs[0, c].legend()
    for (lo, hi), col in zip(LAT_BANDS, band_colors):
        b = TS_W[(TS_W['var'] == v) & (TS_W.band == f'{lo}..{hi}')]
        axs[1, c].plot(b.date, b['diff'], color=col, label=f'{lo}..{hi}')
    axs[1, c].plot(g.date, g['diff'], 'k-', lw=2.5, label='global')
    axs[1, c].axhline(0, color='k', ls=':', lw=1)
    axs[1, c].set_title(f'{v}: allsp − OL6 by latitude band')
    axs[1, c].set_ylabel(f'Δ{v} [m³/m³]')
    axs[1, c].legend(fontsize=8, ncol=2)
    axs[1, c].tick_params(axis='x', rotation=45)
fig.suptitle(f'Daily-mean soil moisture, {START}–{END} (0.5° tavg24, common land cells)', fontsize=14)
fig.savefig(FIG_DIR / 'ts_states_global_and_bands.png', dpi=150)
plt.show()

## 4. Maps at 0.5°

Robinson projection, south of 60S cut off, grey land under the data (so cells with no value show grey),
following the repo map conventions. Difference maps use a discrete **brown–green** (`BrBG`) scale centred on a
white zero bin: **green = allsp wetter than OL6**, brown = drier.

- **4a** The OL6 monthly mean, for context (where it is wet or dry to start with).
- **4b** allsp − OL6, monthly mean: the average effect of DA over May.
- **4c** allsp − OL6 on 31 May only: the state at the end of the month. Where 4c is stronger than 4b,
  the difference has been building up.

In [ ]:
def robinson_panels(fields, titles, levels, cmap, cbar_label, fname, suptitle, extend='both'):
    n = len(fields)
    ncol = 2
    nrow = int(np.ceil(n / ncol))
    fig, axs = plt.subplots(nrow, ncol, figsize=(15, 4.6 * nrow + 0.8),
                            subplot_kw={'projection': ccrs.Robinson()}, constrained_layout=True)
    axs = np.atleast_1d(axs).ravel()
    cm = plt.get_cmap(cmap)
    norm = mcolors.BoundaryNorm(levels, cm.N, extend=extend)
    for ax, field, title in zip(axs, fields, titles):
        ax.set_extent([-180, 180, MAP_LAT_MIN, 90], crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.LAND, facecolor='lightgray', zorder=0)
        mesh = ax.pcolormesh(LON_E, LAT_E, np.where(VALID, field, np.nan), cmap=cm, norm=norm,
                             transform=ccrs.PlateCarree(), rasterized=True, zorder=1)
        ax.coastlines(linewidth=0.3, zorder=2)
        w = np.where(VALID & np.isfinite(field), W2D, 0.0)
        ax.set_title(f'{title}  (area-wtd mean {np.nansum(field * w) / w.sum():.4f})', fontsize=11)
    for ax in axs[n:]:
        ax.set_visible(False)
    fig.colorbar(mesh, ax=axs[:n].tolist(), orientation='horizontal', shrink=0.5, pad=0.02, label=cbar_label)
    fig.suptitle(suptitle, fontsize=14)
    fig.savefig(FIG_DIR / fname, dpi=150)
    plt.show()


with np.errstate(invalid='ignore'), __import__('warnings').catch_warnings():
    __import__('warnings').simplefilter('ignore', RuntimeWarning)   # ocean cells are NaN on every day
    MEAN = {name: {v: np.nanmean(DAILY[name][v], axis=0) for v in VARS} for name in RUNS}

# 4a: OL6 monthly mean
robinson_panels([MEAN[OL_RUN][v] for v in VARS], [f'{OL_RUN} {v}' for v in VARS],
                levels=[0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.5, 0.6, 0.8],
                cmap='YlGnBu', extend='max', cbar_label='soil moisture [m³/m³]',
                fname='map_OL6_monthly_mean.png', suptitle=f'{OL_RUN} monthly mean, {START}–{END}')

DIFF_LEVELS = [-0.1, -0.05, -0.03, -0.02, -0.01, -0.005, 0.005, 0.01, 0.02, 0.03, 0.05, 0.1]

# 4b: allsp - OL6, monthly mean
robinson_panels([MEAN[DA_RUN][v] - MEAN[OL_RUN][v] for v in VARS], [f'allsp − OL6 {v}' for v in VARS],
                levels=DIFF_LEVELS, cmap='BrBG', cbar_label='allsp − OL6 [m³/m³]  (green = DA wetter)',
                fname='map_diff_monthly_mean.png', suptitle=f'allsp − OL6, monthly mean, {START}–{END}')

# 4c: allsp - OL6 on the last day
robinson_panels([DAILY[DA_RUN][v][-1] - DAILY[OL_RUN][v][-1] for v in VARS],
                [f'allsp − OL6 {v}, {DATES[-1].date()}' for v in VARS],
                levels=DIFF_LEVELS, cmap='BrBG', cbar_label='allsp − OL6 [m³/m³]  (green = DA wetter)',
                fname='map_diff_last_day.png', suptitle=f'allsp − OL6, daily mean of {DATES[-1].date()}')

## 5. allsp only, native resolution: where DA adds and removes water

At every assimilation time (every 3 h), `inst3_1d_lndfcstana_Nt` holds the ensemble-mean state before
(`*_FCST`) and after (`*_ANA`) the update, per land tile. `ANA − FCST` is the **analysis increment**. Summed
over all cycles of the month, it is the total water DA put into (+) or took out of (−) each tile's surface layer
and root zone, in m³/m³.

This has no OL6 counterpart (OL6 has no analysis), but it is available at full resolution. Tile sums are then
averaged to **cube cells** with `frac_cell` weights, the same first step MAPL uses for the gridded output. They are
drawn on the **true cube-cell shapes** from the BCS cell raster, with no bilinear step. See
`cf0360_grid_and_tiles.ipynb`, sections 2 and 7, for the geometry. The small helpers below repeat that code so this
notebook stands alone.

Compared with the 0.5° difference in 4b, this map is sharper, and it isolates the direct DA effect. The
allsp − OL6 difference also contains how the model propagates the increments afterwards (drainage,
evaporation, runoff).

In [ ]:
# --- tile table (land tiles first; tile k <-> row k-1) and cube-cell index of each land tile
with nc.Dataset(BCS_GEOM / 'CF0360x6C_CF0360x6C-Pfafstetter.nc4') as d:
    typ = np.asarray(d['typ'][:])
    land = typ == 100
    i_indg = np.asarray(d['i_indg'][:])[land]
    j_indg = np.asarray(d['j_indg'][:])[land]
    frac_cell = np.asarray(d['frac_cell'][:])[land]
N_LAND = int(land.sum())
N_CELL = 6 * N_FACE * N_FACE
LAND_CELL = (j_indg - 1) * N_FACE + i_indg - 1        # 0-based cube-cell index


def tiles_to_cells(values):
    # frac_cell-weighted mean of land-tile values per cube cell (MAPL T2G); NaN where no valid land tile
    ok = np.isfinite(values)
    num = np.bincount(LAND_CELL[ok], weights=frac_cell[ok] * values[ok], minlength=N_CELL)
    den = np.bincount(LAND_CELL[ok], weights=frac_cell[ok], minlength=N_CELL)
    out = np.full(N_CELL, np.nan)
    out[den > 0] = num[den > 0] / den[den > 0]
    return out


# --- cube-cell raster, every 20th pixel of the 30-arcsec BCS raster (~0.17 deg); values = 1-based cell number
STRIDE, RES = 20, 1.0 / 120.0
mm = np.memmap(BCS_GEOM / 'rst' / 'CF0360x6C.rst', dtype='<i4', mode='r').reshape(21600, 43202)
CELL_RASTER = np.array(mm[::STRIDE, 1:-1:STRIDE])      # skip Fortran record markers; rows S -> N
CR_LAT_E = -90 + RES * STRIDE * np.arange(CELL_RASTER.shape[0] + 1)
CR_LON_E = -180 + RES * STRIDE * np.arange(CELL_RASTER.shape[1] + 1)
print(f'{N_LAND:,} land tiles, cell raster {CELL_RASTER.shape}')

### 5a. Sum the increments over the month (cached)

For each 3-hourly file in [START, END] we add up, per tile, `ANA − FCST` for SFMC and RZMC, plus the absolute
value (to see where DA is *active* even when positive and negative increments cancel). We also keep the
number of cycles in which the tile had a non-zero increment. The global area-weighted mean increment per
cycle is stored as a time series.

In [ ]:
incr_cache = CACHE_DIR / 'allsp_state_increment_sums.npz'
incr_ts_cache = CACHE_DIR / 'allsp_state_increment_ts.csv'

if incr_cache.exists() and incr_ts_cache.exists() and not FORCE_RECOMPUTE:
    INC = dict(np.load(incr_cache))
    INC_TS = pd.read_csv(incr_ts_cache, parse_dates=['time'])
    print('using cached increment sums')
else:
    t0 = time.time()
    exp = RUNS[DA_RUN]
    files = sorted(glob.glob(str(ROOT / exp / 'output' / DOMAIN / 'cat' / 'ens_avg' / 'Y*' / 'M*'
                                 / f'{exp}.inst3_1d_lndfcstana_Nt.*.nc4')))
    files = [f for f in files if START <= os.path.basename(f).split('.')[-2][:8] <= END]
    INC = {f'{k}_{v}': np.zeros(N_LAND) for v in VARS for k in ['sum', 'abs', 'nact']}
    ts = []
    for f in files:
        stamp = os.path.basename(f).split('.')[-2]
        with nc.Dataset(f) as d:
            row = {'time': pd.to_datetime(stamp, format='%Y%m%d_%H%Mz')}
            for v in VARS:
                inc = np.ma.filled(d[f'{v}_ANA'][0].astype(np.float64) - d[f'{v}_FCST'][0].astype(np.float64),
                                   np.nan)
                inc = np.where(np.isfinite(inc), inc, 0.0)
                INC[f'sum_{v}'] += inc
                INC[f'abs_{v}'] += np.abs(inc)
                INC[f'nact_{v}'] += inc != 0
                row[f'mean_incr_{v}'] = inc.mean()        # unweighted over tiles (for a quick time series)
            ts.append(row)
    INC_TS = pd.DataFrame(ts)
    np.savez_compressed(incr_cache, **INC)
    INC_TS.to_csv(incr_ts_cache, index=False)
    print(f'{len(files)} cycles summed in {time.time() - t0:.0f} s')
N_CYCLES = len(INC_TS)

### 5b. Maps on cube cells: net and absolute summed increments

- **Top:** net summed increment (`Σ(ANA − FCST)`) for SFMC and RZMC. Green = DA added water over the month.
- **Bottom:** summed absolute increment (`Σ|ANA − FCST|`): how much DA moved the state, in either direction.

Cube-cell values are `frac_cell`-weighted means of the tile sums, drawn on the true cell shapes.

In [ ]:
def cube_panels(cell_fields, titles, levels_list, cmaps, cbar_labels, fname, suptitle):
    fig, axs = plt.subplots(2, 2, figsize=(15, 10), subplot_kw={'projection': ccrs.Robinson()},
                            constrained_layout=True)
    for ax, field, title, lev, cmap, lab in zip(axs.ravel(), cell_fields, titles, levels_list, cmaps, cbar_labels):
        cm = plt.get_cmap(cmap)
        extend = 'both' if lev[0] < 0 else 'max'
        norm = mcolors.BoundaryNorm(lev, cm.N, extend=extend)
        ax.set_extent([-180, 180, MAP_LAT_MIN, 90], crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.LAND, facecolor='lightgray', zorder=0)
        mesh = ax.pcolormesh(CR_LON_E, CR_LAT_E, field[CELL_RASTER - 1], cmap=cm, norm=norm,
                             transform=ccrs.PlateCarree(), rasterized=True, zorder=1)
        ax.coastlines(linewidth=0.3, zorder=2)
        ax.set_title(title, fontsize=11)
        fig.colorbar(mesh, ax=ax, orientation='horizontal', shrink=0.7, pad=0.02, label=lab)
    fig.suptitle(suptitle, fontsize=14)
    fig.savefig(FIG_DIR / fname, dpi=150)
    plt.show()


cell = {k: tiles_to_cells(v) for k, v in INC.items()}
# Surface increments are several times larger than root-zone ones, so each variable gets its own scale
NET_LEVELS = {'SFMC': [-1, -0.5, -0.2, -0.1, -0.03, -0.01, 0.01, 0.03, 0.1, 0.2, 0.5, 1],
              'RZMC': [-0.2, -0.1, -0.05, -0.03, -0.01, -0.003, 0.003, 0.01, 0.03, 0.05, 0.1, 0.2]}
ABS_LEVELS = {'SFMC': [0, 0.1, 0.2, 0.5, 1, 1.5, 2, 3, 5],
              'RZMC': [0, 0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.5, 1.0]}
cube_panels([cell['sum_SFMC'], cell['sum_RZMC'], cell['abs_SFMC'], cell['abs_RZMC']],
            ['net Σ increment SFMC', 'net Σ increment RZMC', 'Σ |increment| SFMC', 'Σ |increment| RZMC'],
            [NET_LEVELS['SFMC'], NET_LEVELS['RZMC'], ABS_LEVELS['SFMC'], ABS_LEVELS['RZMC']],
            ['BrBG', 'BrBG', 'magma_r', 'magma_r'],
            ['Σ(ANA − FCST) [m³/m³]', 'Σ(ANA − FCST) [m³/m³]', 'Σ|ANA − FCST| [m³/m³]', 'Σ|ANA − FCST| [m³/m³]'],
            'allsp_incr_sums_cube_cells.png',
            f'allsp analysis increments summed over {N_CYCLES} cycles, {START}–{END} (CF0360 cube cells)')

### 5c. Time series of the mean increment per cycle

The mean over all land tiles of `ANA − FCST` at each cycle (unweighted by tile area; a quick diagnostic of
whether DA adds or removes water on average, and whether that depends on the time of day, i.e. on which
sensors pass).

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(15, 7), sharex=True, constrained_layout=True)
for ax, v in zip(axs, VARS):
    ax.plot(INC_TS.time, INC_TS[f'mean_incr_{v}'], '-', lw=0.8, color='0.3')
    ax.axhline(0, color='k', ls=':', lw=1)
    ax.set_ylabel(f'mean ΔSM per cycle [m³/m³]')
    ax.set_title(f'{v}: land-tile mean analysis increment per cycle '
                 f'(month total {INC_TS[f"mean_incr_{v}"].sum():+.4f})')
axs[-1].tick_params(axis='x', rotation=45)
fig.savefig(FIG_DIR / 'ts_allsp_mean_increment_per_cycle.png', dpi=150)
plt.show()

by_hour = INC_TS.assign(hour=INC_TS.time.dt.hour).groupby('hour')[[f'mean_incr_{v}' for v in VARS]].mean()
print('mean increment per cycle by synoptic hour (UTC):')
print(by_hour.to_string(float_format=lambda x: f'{x:+.2e}'))

## 6. Notes

- The allsp − OL6 comparison (sections 3–4) is limited to 0.5° because OL6 wrote only `tavg24_2d_lnd_Nx`. For a
  tile- or cube-cell-level comparison in future runs, turn on a tile-space (`_Nt`) collection in the OL as well, or
  comment out `grid_label` in `HISTORY.rc` to get cube-face output.
- OL6 has 6 members and allsp 24. Both files hold the ensemble mean, so part of a small-scale allsp − OL6
  difference can come from the different ensemble sizes, not from DA. The OL6 vs 24-member OL check in the
  write-up found OL6 a fair reference in obs space (5/1–5/10).